# conv-padding-zero — worked example 2: Pad the channel axis, not the spatial axis

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-padding-zero`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The allocate-zero-buffer-then-assign-interior pattern is not specific to spatial dims — it works on any axis. Here we zero-pad the *channel* dimension of a `(B, IC, W)` tensor, a trick used to match channel counts in residual connections. The spatial width is left unchanged.

## Worked solution

Goal: take `(B, IC, W)` and produce `(B, before + IC + after, W)`, where the inserted channels are all zero.

1. **Shape read.** `B, IC, W = x.shape`.
2. **Allocate.** `out = x.new_zeros(B, before + IC + after, W)`. The new channel-axis length is `before + IC + after`; the spatial axis `W` is copied unchanged. `new_zeros` inherits dtype/device.
3. **Assign the interior on the channel axis.** The real channels belong at index `[before : before + IC]` of dim 1. Because dim 1 is not the last axis, we index it explicitly: `out[:, before : before + IC, :] = x`.
4. **Why it works.** The leading `before` channels and trailing `after` channels were never written, so they stay zero — exactly the zero feature-maps a residual add needs. The middle block is a faithful copy of `x`'s channels.

In [ ]:
def pad_channels(x: Tensor, before: int, after: int) -> Tensor:
    B, IC, W = x.shape
    out = x.new_zeros(B, before + IC + after, W)
    out[:, before : before + IC, :] = x
    return out

t.manual_seed(0)
x = t.ones(1, 3, 4)
y = pad_channels(x, 1, 2)
print(y.shape)
print('padded channels zero:', bool((y[:, 0] == 0).all()) and bool((y[:, -2:] == 0).all()))
print('real channels one:', bool((y[:, 1:4] == 1).all()))